# Comparing models without rainfall

Compare each model with its usual six features and with `rainfall_inches` removed. Only cleaned training data and cross-validation are used; the held-out Imelda test data is not loaded.

## 1. Load training data and verify the default behavior

The optional feature-removal setting defaults to no removals. Check that the existing logistic-regression area-fold results are unchanged before running the comparison.

In [1]:
import sys
from pathlib import Path
from time import perf_counter

project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "config.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the FloodSense project root from the notebook working directory.")
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import CATEGORICAL_FEATURES, NUMERIC_FEATURES, SEED
from src.data import load_clean
from src.evaluate import RESULTS_PATH, compute_metrics, cross_validate, save_results
from src.features import area_group_splits, build_preprocessor, get_xy, leave_one_storm_out_splits
from src.models import MODEL_REGISTRY
from src.threshold import pick_threshold

df = load_clean()
area_splits = list(area_group_splits(df, n_splits=5))
storm_splits = list(leave_one_storm_out_splits(df, drop_overlap_areas=False))
baseline_summary, _ = cross_validate(
    MODEL_REGISTRY["logistic_regression"], df, area_splits,
    name="logistic_regression", split_name="area_group",
)
baseline_mean = baseline_summary.loc[baseline_summary["fold"].eq("mean")].iloc[0]
assert np.isclose(baseline_mean["recall"], 0.789325858096668)
assert np.isclose(baseline_mean["roc_auc"], 0.7841340551596213)
print(f"Default logistic area recall: {baseline_mean['recall']:.3f}; ROC-AUC: {baseline_mean['roc_auc']:.3f}")
print(f"Area folds: {len(area_splits)}; storm folds: {len(storm_splits)}; seed: {SEED}")

Cleaned training data shape: (66514, 14)
Missing values by column:
year                   0
storm_name             0
storm_date             0
lat                    0
lon                    0
weight                 0
elevation_m            0
slope_degrees          0
distance_to_bayou_m    0
paved_pct              0
land_cover_type        0
rainfall_inches        0
flooded                0
grid_id                0


Default logistic area recall: 0.789; ROC-AUC: 0.784
Area folds: 5; storm folds: 3; seed: 42


**My observations:**

## 2. Run all models without rainfall

For each model, run area-group and leave-one-storm-out validation with fresh fold-local preprocessing. The saved model name includes `_no_rain`, so the original result rows remain separate. The timer reports both split runs for each model.

In [2]:
models = (
    "logistic_regression",
    "random_forest",
    "xgboost",
    "lightgbm",
    "mlp",
)
split_sets = (("area_group", area_splits), ("leave_one_storm_out", storm_splits))
no_rain_results = {}
model_times = {}
for model in models:
    started = perf_counter()
    for split_name, splits in split_sets:
        summary, oof = cross_validate(
            MODEL_REGISTRY[model],
            df,
            splits,
            name=f"{model}_no_rain",
            split_name=split_name,
            drop_features=("rainfall_inches",),
        )
        save_results(summary, oof, name=f"{model}_no_rain", split_name=split_name)
        no_rain_results[(model, split_name)] = (summary, oof)
    model_times[model] = perf_counter() - started
    print(f"{model} without rainfall, both split designs: {model_times[model]:.1f} seconds")
print("All no-rainfall cross-validation runs completed.")

Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_logistic_regression_no_rain_area_group.csv


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_logistic_regression_no_rain_leave_one_storm_out.csv
logistic_regression without rainfall, both split designs: 0.7 seconds


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_random_forest_no_rain_area_group.csv


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_random_forest_no_rain_leave_one_storm_out.csv
random_forest without rainfall, both split designs: 39.8 seconds


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_xgboost_no_rain_area_group.csv


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_xgboost_no_rain_leave_one_storm_out.csv
xgboost without rainfall, both split designs: 11.0 seconds


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_lightgbm_no_rain_area_group.csv


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_lightgbm_no_rain_leave_one_storm_out.csv
lightgbm without rainfall, both split designs: 8.5 seconds


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_mlp_no_rain_area_group.csv


Saved results to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\results.csv
Saved OOF predictions to C:\Users\ssaifudd\Downloads\AI-Flood Project\reports\oof_mlp_no_rain_leave_one_storm_out.csv
mlp without rainfall, both split designs: 244.0 seconds
All no-rainfall cross-validation runs completed.


**My observations:**

## 3. Compare with-rainfall and no-rainfall results

This table uses the saved mean rows for the original models and their new `_no_rain` versions.

In [3]:
all_results = pd.read_csv(RESULTS_PATH)
models_with_rain = models
models_without_rain = [f"{model}_no_rain" for model in models]
comparison = all_results.loc[
    all_results["model"].isin([*models_with_rain, *models_without_rain])
    & all_results["split"].isin(("area_group", "leave_one_storm_out"))
    & all_results["fold"].eq("mean"),
    ["model", "split", "recall", "precision", "f1", "roc_auc", "pr_auc"],
].copy()
comparison["features"] = np.where(comparison["model"].str.endswith("_no_rain"), "without_rainfall", "with_rainfall")
comparison["model"] = comparison["model"].str.removesuffix("_no_rain")
expected_rows = len(models) * 2 * 2
if len(comparison) != expected_rows:
    raise ValueError(f"Expected {expected_rows} comparison rows, found {len(comparison)}.")
comparison = comparison[["model", "features", "split", "recall", "precision", "f1", "roc_auc", "pr_auc"]]
comparison = comparison.sort_values(["split", "model", "features"]).reset_index(drop=True)
display(comparison)

,model,features,split,recall,precision,f1,roc_auc,pr_auc
0,lightgbm,with_rainfall,area_group,0.857554,0.813940,0.835147,0.900738,0.879999
1,lightgbm,without_rainfall,area_group,0.852761,0.801182,0.826129,0.891287,0.867742
2,logistic_regression,with_rainfall,area_group,0.789326,0.704909,0.744666,0.784134,0.744711
3,logistic_regression,without_rainfall,area_group,0.790242,0.701920,0.743392,0.780928,0.743125
4,mlp,with_rainfall,area_group,0.832390,0.764168,0.796669,0.862315,0.841043
5,mlp,without_rainfall,area_group,0.829115,0.751848,0.788396,0.853027,0.831565
6,random_forest,with_rainfall,area_group,0.823937,0.809886,0.816790,0.887299,0.866076
7,random_forest,without_rainfall,area_group,0.788273,0.794367,0.791242,0.865809,0.839729
8,xgboost,with_rainfall,area_group,0.856591,0.812761,0.834053,0.899914,0.879417
9,xgboost,without_rainfall,area_group,0.844528,0.797900,0.820513,0.886937,0.863240


**My observations:**

## 4. Compare per-storm OOF scores

Use saved OOF predictions to calculate recall and ROC-AUC at 0.5 for each storm and overall, with and without rainfall. Row numbers map each OOF prediction to the storm in cleaned training data.

In [4]:
storm_order = ["Hurricane Harvey", "Memorial Day Flood", "Tax Day Flood", "Overall"]
per_storm_rows = []
for model in models:
    for feature_label, saved_model in (("with_rainfall", model), ("without_rainfall", f"{model}_no_rain")):
        oof_path = project_root / "reports" / f"oof_{saved_model}_leave_one_storm_out.csv"
        oof = pd.read_csv(oof_path)
        indices = oof["row_index"].astype(int).to_numpy()
        oof["storm_name"] = df.iloc[indices]["storm_name"].to_numpy()
        if not np.array_equal(df.iloc[indices]["flooded"].to_numpy(), oof["y_true"].to_numpy()):
            raise ValueError(f"OOF labels do not match cleaned row indices for {saved_model}.")
        groups = [("Overall", oof), *list(oof.groupby("storm_name", sort=False))]
        for storm_name, group in groups:
            metrics = compute_metrics(group["y_true"], group["y_prob"], threshold=0.5)
            per_storm_rows.append(
                {
                    "model": model,
                    "features": feature_label,
                    "storm": storm_name,
                    "recall": metrics["recall"],
                    "roc_auc": metrics["roc_auc"],
                }
            )
per_storm_table = pd.DataFrame(per_storm_rows)
per_storm_table["storm"] = pd.Categorical(per_storm_table["storm"], storm_order, ordered=True)
display(per_storm_table.sort_values(["model", "storm", "features"]).reset_index(drop=True))

,model,features,storm,recall,roc_auc
0,lightgbm,with_rainfall,Hurricane Harvey,0.123384,0.784411
1,lightgbm,without_rainfall,Hurricane Harvey,0.878983,0.899057
2,lightgbm,with_rainfall,Memorial Day Flood,0.717949,0.847604
3,lightgbm,without_rainfall,Memorial Day Flood,0.895617,0.906751
4,lightgbm,with_rainfall,Tax Day Flood,0.606526,0.852460
5,lightgbm,without_rainfall,Tax Day Flood,0.878281,0.898621
6,lightgbm,with_rainfall,Overall,0.474245,0.812099
7,lightgbm,without_rainfall,Overall,0.884141,0.901318
8,logistic_regression,with_rainfall,Hurricane Harvey,0.967210,0.738911
9,logistic_regression,without_rainfall,Hurricane Harvey,0.789572,0.781208


**My observations:**

## 5. Check area-selected no-rainfall cutoffs on storm folds

Choose the highest area-fold cutoff reaching 0.85 recall for each no-rainfall model. Apply it unchanged to overall and per-storm OOF predictions.

In [5]:
cutoff_rows = []
for model in models:
    area_oof = no_rain_results[(model, "area_group")][1]
    storm_oof = no_rain_results[(model, "leave_one_storm_out")][1].copy()
    threshold = pick_threshold(area_oof, target_recall=0.85)
    indices = storm_oof["row_index"].astype(int).to_numpy()
    storm_oof["storm_name"] = df.iloc[indices]["storm_name"].to_numpy()
    scopes = [("Overall", storm_oof), *list(storm_oof.groupby("storm_name", sort=False))]
    for scope, group in scopes:
        metrics = (
            compute_metrics(group["y_true"], group["y_prob"], threshold=threshold)
            if threshold is not None
            else {"recall": np.nan, "precision": np.nan}
        )
        cutoff_rows.append(
            {
                "model": model,
                "area_chosen_cutoff": threshold,
                "scope": scope,
                "storm_recall": metrics["recall"],
                "storm_precision": metrics["precision"],
            }
        )
cutoff_table = pd.DataFrame(cutoff_rows)
display(cutoff_table.sort_values(["model", "scope"]).reset_index(drop=True))

,model,area_chosen_cutoff,scope,storm_recall,storm_precision
0,lightgbm,0.50,Hurricane Harvey,0.878983,0.809102
1,lightgbm,0.50,Memorial Day Flood,0.895617,0.802706
2,lightgbm,0.50,Overall,0.884141,0.805155
3,lightgbm,0.50,Tax Day Flood,0.878281,0.803454
4,logistic_regression,0.44,Hurricane Harvey,0.855573,0.686233
5,logistic_regression,0.44,Memorial Day Flood,0.865703,0.669550
6,logistic_regression,0.44,Overall,0.855927,0.677333
7,logistic_regression,0.44,Tax Day Flood,0.846638,0.675852
8,mlp,0.47,Hurricane Harvey,0.844841,0.751993
9,mlp,0.47,Memorial Day Flood,0.861884,0.738412


**My observations:**

## 6. No-rainfall Random Forest feature importance

Fit a no-rainfall Random Forest on all cleaned training rows and show its five largest built-in importances. This is a rough guide only, not a final explanation or causal result.

In [6]:
drop_features = ("rainfall_inches",)
X, y = get_xy(df, drop_features=drop_features)
preprocessor = build_preprocessor(drop_features=drop_features)
X_transformed = preprocessor.fit_transform(X)
random_forest = MODEL_REGISTRY["random_forest"]()
random_forest.fit(X_transformed, y)
feature_names = preprocessor.get_feature_names_out()
top_indices = np.argsort(random_forest.feature_importances_)[-5:][::-1]
top_no_rain_features = pd.DataFrame(
    {"feature": feature_names[top_indices], "importance": random_forest.feature_importances_[top_indices]}
)
display(top_no_rain_features)
print("These importance values are a rough guide only.")

,feature,importance
0,numeric__elevation_m,0.352111
1,numeric__slope_degrees,0.264802
2,numeric__distance_to_bayou_m,0.179319
3,numeric__paved_pct,0.157541
4,"categorical__land_cover_type_Developed, Low In...",0.017395


These importance values are a rough guide only.


**My observations:**